### Spatial feature Extraction 
Creating the dataset of the functional nuclear powerplants (412) with 3,000 fake sites that serves at the negative instances for the ML model thus total of 3412 instances are used to train an ML model, the spatial features that are to be used for the final dataset are :
* ID (type : Int)
* Latitude (type : Continuous)
* longitude (type : Continuous)
* Region (type : Char)
* seismic_pga (type : Continuous)
* dist_to_water
* dist_to_fault (type : binary)
* population
* is_protected_area (type : Binary)
* dist_to_volcano
* border_cooperation_score
* target (type : Binary)

In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
import reverse_geocoder as rg
import pycountry_convert as pc
import rasterio

In [2]:
# Cleaning the functional powerplant dataset 
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/operating_plants.csv")
print(df.columns)
# Dropping unnecessary columns
df = df.drop(columns=['Id', 'Name', 'CountryCode',
       'Status', 'ReactorType', 'ReactorModel', 'ConstructionStartAt',
       'OperationalFrom', 'OperationalTo', 'Capacity', 'LastUpdatedAt',
       'Source', 'IAEAId'])

# creating an Id column
df.insert(0, "Id", range(len(df)))


Index(['Id', 'Name', 'Latitude', 'Longitude', 'Country', 'CountryCode',
       'Status', 'ReactorType', 'ReactorModel', 'ConstructionStartAt',
       'OperationalFrom', 'OperationalTo', 'Capacity', 'LastUpdatedAt',
       'Source', 'IAEAId'],
      dtype='object')


In [3]:
# creaing a region column
regions = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica"
}
def latlon_to_continent(lat, lon):
    try:
        result = rg.search((lat, lon))[0]
        country_code = result["cc"]
        cont_code = pc.country_alpha2_to_continent_code(country_code)
        print(f"Latitude: {lat}, Longitude: {lon} -> Country Code: {country_code}, Continent Code: {cont_code}")
        return regions[cont_code]
    except:
        return None

df["Region"] = df.apply(
    lambda row: latlon_to_continent(row["Latitude"], row["Longitude"]),
    axis=1
)

Loading formatted geocoded file...
Latitude: 69.709579, Longitude: 170.30625 -> Country Code: RU, Continent Code: EU
Latitude: 69.709579, Longitude: 170.30625 -> Country Code: RU, Continent Code: EU
Latitude: 39.807, Longitude: -5.698 -> Country Code: ES, Continent Code: EU
Latitude: 39.807, Longitude: -5.698 -> Country Code: ES, Continent Code: EU
Latitude: -23.008, Longitude: -44.457 -> Country Code: BR, Continent Code: SA
Latitude: -23.008, Longitude: -44.457 -> Country Code: BR, Continent Code: SA
Latitude: 35.31, Longitude: -93.23 -> Country Code: US, Continent Code: NA
Latitude: 35.31, Longitude: -93.229 -> Country Code: US, Continent Code: NA
Latitude: 40.182, Longitude: 44.147 -> Country Code: AM, Continent Code: AS
Latitude: 41.202, Longitude: 0.571 -> Country Code: ES, Continent Code: EU
Latitude: 41.202, Longitude: 0.571 -> Country Code: ES, Continent Code: EU
Latitude: -33.967, Longitude: -59.209 -> Country Code: AR, Continent Code: SA
Latitude: -33.967, Longitude: -59.209 

In [ ]:
# Creating target column
df["Target"] = 1.0

# creating seismic_pga, dist_to_water, dist_to_fault, population, is_protected_area, dist_to_volcano, border_cooperation_score
# using rasterio to read the raster files and extract the values for each plant location
RASTER_DIR   = "/Users/mohammadbilal/Documents/Projects/N-Project/data/processed/rasters"
OUTPUT_CSV = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/ml_dataset.csv"
RASTERS = {
    "seismic_pga":      f"{RASTER_DIR}/seismic_data.tif",
    "dist_to_water":    f"{RASTER_DIR}/distance_to_water.tif",
    "dist_to_fault":    f"{RASTER_DIR}/fault_exclusion_mask.tif",
    "population":       f"{RASTER_DIR}/population_data.tif",
    "is_protected":     f"{RASTER_DIR}/protected_areas_mask.tif",
    "dist_to_volcano":  f"{RASTER_DIR}/volcano_data.tif",
    "border_distance":  f"{RASTER_DIR}/distance_to_border.tif",
}

def sample_point(lon, lat, raster_handles):
    values = {}
    for name, src in raster_handles.items():
        try:
            row, col = src.index(lon, lat)
            window = rasterio.windows.Window(col, row, 1, 1)
            data = src.read(1, window=window)
            val = float(data[0, 0])
            nodata = src.nodata
            if nodata is not None:
                # For float max nodata, use tolerance; for everything else exact match
                if abs(nodata) > 1e307:
                    if abs(val - nodata) < 1e300:
                        val = np.nan
                else:
                    if val == nodata:
                        val = np.nan
        except Exception:
            val = np.nan
        values[name] = val
    return values

raster_handles = {name: rasterio.open(path) for name, path in RASTERS.items()}

# Sample rasters at every point
records = []
skipped = 0

for _, row in df.iterrows():
    vals = sample_point(row["Longitude"], row["Latitude"], raster_handles)
 
    records.append({
        "Id":        row["Id"],
        "Latitude":  row["Latitude"],
        "Longitude": row["Longitude"],
        "Country":   row["Country"],
        "Region":    row["Region"],
        "Target":    row["Target"],
        **vals,
    })

# Close rasters
for src in raster_handles.values():
    src.close()

ml_df = pd.DataFrame(records)
ml_df.drop(columns=["Country"], inplace=True)
ml_df.to_csv(OUTPUT_CSV, index=False)
